# Repeated validation of the selected pooled model

This notebook validates the selected source/class-balanced pooled Inception model over five participant folds and three random seeds. Each outer validation fold is untouched. Within each outer training fold, an inner participant split is used for early stopping, Platt calibration, and threshold selection.

The final reported outer-fold metrics therefore use thresholds and calibration learned only from participants inside the corresponding training fold.

In [1]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, brier_score_loss, f1_score, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit
from torch import nn
from torch.utils.data import DataLoader, Dataset

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED = PROJECT_ROOT / 'data' / 'processed'; INTERIM = PROJECT_ROOT / 'data' / 'interim'
MAG_PATH = PROCESSED / 'validated_acceleration_magnitude_windows_float32.npy'
METADATA_PATH = PROCESSED / 'validated_window_metadata.csv'; SPLITS_PATH = INTERIM / 'participant_splits.csv'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(4); EPOCHS = 4; BATCH_SIZE = 128
metadata = pd.read_csv(METADATA_PATH); metadata['label_binary'] = metadata['label'].map({'healthy': 0, 'stroke': 1}).astype(int)
splits = pd.read_csv(SPLITS_PATH); magnitude_windows = np.load(MAG_PATH, mmap_mode='r')
print('Device:', DEVICE, '| windows:', magnitude_windows.shape)
print(metadata.groupby(['dataset_id', 'label']).participant_key.nunique())

Device: cuda | windows: (18511, 500, 3)
dataset_id    label  
felius_2024   healthy     34
              stroke     129
voisard_2025  healthy     72
              stroke      49
Name: participant_key, dtype: int64


In [2]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def outer_indices(fold):
    role_map = splits[splits.fold.eq(fold)].set_index('participant_key').role
    roles = metadata.participant_key.map(role_map)
    return np.flatnonzero(roles.eq('training').to_numpy()), np.flatnonzero(roles.eq('validation').to_numpy())

def inner_split(train_indices, seed):
    participants = metadata.iloc[train_indices][['participant_key', 'dataset_id', 'label_binary']].drop_duplicates()
    participants['stratum'] = participants.dataset_id + '_' + participants.label_binary.astype(str)
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
    fit_pos, cal_pos = next(splitter.split(participants, participants.stratum))
    fit_keys = set(participants.iloc[fit_pos].participant_key); cal_keys = set(participants.iloc[cal_pos].participant_key)
    fit_indices = metadata.index[metadata.participant_key.isin(fit_keys)].to_numpy()
    cal_indices = metadata.index[metadata.participant_key.isin(cal_keys)].to_numpy()
    return fit_indices, cal_indices

def pooled_stats(indices):
    total = np.zeros(3, dtype='float64'); total_sq = np.zeros(3, dtype='float64'); count = 0
    for start in range(0, len(indices), 512):
        batch = np.asarray(magnitude_windows[indices[start:start + 512]], dtype='float32')
        total += batch.sum(axis=(0, 1)); total_sq += np.square(batch).sum(axis=(0, 1)); count += batch.shape[0] * batch.shape[1]
    mean = total / count; std = np.sqrt(np.maximum(total_sq / count - mean ** 2, 1e-8))
    return mean.astype('float32'), std.astype('float32')

def source_class_balanced_weights(indices):
    frame = metadata.iloc[indices].copy(); window_counts = frame.groupby('participant_key').size()
    cell_counts = frame.groupby(['dataset_id', 'label_binary']).participant_key.nunique()
    participant_weight = frame.participant_key.map(1.0 / window_counts).to_numpy()
    cell_weight = np.array([1.0 / cell_counts[(r.dataset_id, r.label_binary)] for r in frame.itertuples()])
    weights = participant_weight * cell_weight
    return (weights / weights.mean()).astype('float32')

In [3]:
class GaitDataset(Dataset):
    def __init__(self, indices, mean, std, weights=None):
        self.indices = np.asarray(indices, dtype='int64'); self.mean = mean.reshape(1, 3); self.std = std.reshape(1, 3)
        self.weights = np.ones(len(self.indices), dtype='float32') if weights is None else weights
    def __len__(self): return len(self.indices)
    def __getitem__(self, item):
        index = int(self.indices[item]); signal = np.asarray(magnitude_windows[index], dtype='float32')
        signal = ((signal - self.mean) / self.std).T.copy()
        return torch.from_numpy(signal), torch.tensor(float(metadata.iloc[index].label_binary)), torch.tensor(float(self.weights[item])), torch.tensor(index)

class InceptionBlock(nn.Module):
    def __init__(self, in_channels, out_channels=16):
        super().__init__(); bottleneck = min(32, in_channels)
        self.bottleneck = nn.Conv1d(in_channels, bottleneck, 1, bias=False)
        self.branches = nn.ModuleList([nn.Conv1d(bottleneck, out_channels, 7, padding=3, bias=False), nn.Conv1d(bottleneck, out_channels, 15, padding=7, bias=False), nn.Conv1d(bottleneck, out_channels, 25, padding=12, bias=False)])
        self.pool_branch = nn.Conv1d(in_channels, out_channels, 1, bias=False); self.bn = nn.BatchNorm1d(out_channels * 4)
        self.residual = nn.Conv1d(in_channels, out_channels * 4, 1, bias=False) if in_channels != out_channels * 4 else nn.Identity()
    def forward(self, x):
        z = self.bottleneck(x); branches = [branch(z) for branch in self.branches]; branches.append(self.pool_branch(nn.functional.max_pool1d(x, 3, stride=1, padding=1)))
        return nn.functional.gelu(self.bn(torch.cat(branches, dim=1)) + self.residual(x))

class InceptionCNN(nn.Module):
    def __init__(self):
        super().__init__(); self.features = nn.Sequential(InceptionBlock(3), nn.MaxPool1d(2), InceptionBlock(64), nn.AdaptiveAvgPool1d(1)); self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.30), nn.Linear(64, 1))
    def forward(self, x): return self.classifier(self.features(x)).squeeze(1)

def predict_logits(model, loader):
    model.eval(); rows = []
    with torch.no_grad():
        for signals, _, _, indices in loader:
            logits = model(signals.to(DEVICE)).cpu().numpy(); rows.extend(zip(indices.numpy(), logits))
    return np.array([p for _, p in rows], dtype='float32')

def participant_logits(indices, logits, fold, seed):
    frame = metadata.iloc[np.asarray(indices)].copy(); frame['logit'] = logits
    frame = frame.groupby(['participant_key', 'dataset_id', 'label_binary'], as_index=False).logit.mean()
    frame['fold'] = fold; frame['seed'] = seed; return frame

def ece(y, p, bins=10):
    edges = np.linspace(0, 1, bins + 1); result = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (p >= lo) & (p < hi if hi < 1 else p <= hi)
        if mask.any(): result += mask.mean() * abs(p[mask].mean() - y[mask].mean())
    return float(result)

In [4]:
def fit_outer_model(fold, seed):
    set_seed(seed); train_indices, test_indices = outer_indices(fold); fit_indices, cal_indices = inner_split(train_indices, seed + fold)
    mean, std = pooled_stats(fit_indices); weights = source_class_balanced_weights(fit_indices)
    fit_loader = DataLoader(GaitDataset(fit_indices, mean, std, weights), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    cal_loader = DataLoader(GaitDataset(cal_indices, mean, std), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(GaitDataset(test_indices, mean, std), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    model = InceptionCNN().to(DEVICE); optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    best_state = None; best_auc = -np.inf; patience = 2
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for signals, labels, weights_batch, _ in fit_loader:
            optimizer.zero_grad(); logits = model(signals.to(DEVICE))
            loss = (nn.functional.binary_cross_entropy_with_logits(logits, labels.to(DEVICE), reduction='none') * weights_batch.to(DEVICE)).mean(); loss.backward(); optimizer.step()
        cal_logits = predict_logits(model, cal_loader); cal_frame = participant_logits(cal_indices, cal_logits, fold, seed); cal_prob = 1 / (1 + np.exp(-cal_frame.logit.to_numpy()))
        cal_auc = roc_auc_score(cal_frame.label_binary, cal_prob)
        if cal_auc > best_auc:
            best_auc = cal_auc; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}; patience = 2
        else:
            patience -= 1
            if patience == 0: break
    model.load_state_dict(best_state)
    cal_frame = participant_logits(cal_indices, predict_logits(model, cal_loader), fold, seed)
    test_frame = participant_logits(test_indices, predict_logits(model, test_loader), fold, seed)
    calibrator = LogisticRegression(C=1e6, solver='lbfgs').fit(cal_frame[['logit']], cal_frame.label_binary)
    cal_frame['raw_probability'] = 1 / (1 + np.exp(-cal_frame.logit))
    test_frame['raw_probability'] = 1 / (1 + np.exp(-test_frame.logit))
    cal_frame['calibrated_probability'] = calibrator.predict_proba(cal_frame[['logit']])[:, 1]
    test_frame['calibrated_probability'] = calibrator.predict_proba(test_frame[['logit']])[:, 1]
    thresholds = np.unique(np.round(cal_frame.calibrated_probability.to_numpy(), 6)); threshold_scores = [(t, balanced_accuracy_score(cal_frame.label_binary, (cal_frame.calibrated_probability >= t).astype(int))) for t in thresholds]
    threshold = max(threshold_scores, key=lambda x: (x[1], -abs(x[0] - 0.5)))[0]
    test_frame['threshold'] = threshold; test_frame['fold'] = fold; test_frame['seed'] = seed
    return test_frame, threshold

In [5]:
prediction_frames = []; run_rows = []
for seed in [42, 52, 62]:
    for fold in range(5):
        frame, threshold = fit_outer_model(fold, seed); prediction_frames.append(frame)
        for probability_type, probability, cutoff in [('raw', frame.raw_probability, 0.5), ('calibrated', frame.calibrated_probability, threshold)]:
            for scope, part in [('pooled_all', frame), *frame.groupby('dataset_id')]:
                y = part.label_binary.to_numpy(); p = probability.loc[part.index].to_numpy(); pred = (p >= cutoff).astype(int)
                run_rows.append({'seed': seed, 'fold': fold, 'scope': scope, 'probability_type': probability_type, 'threshold': cutoff, 'participants': len(part), 'balanced_accuracy': balanced_accuracy_score(y, pred), 'roc_auc': roc_auc_score(y, p), 'f1': f1_score(y, pred), 'brier': brier_score_loss(y, p), 'ece_10bin': ece(y, p)})
        print('seed', seed, 'fold', fold, 'threshold', round(threshold, 3))
predictions = pd.concat(prediction_frames, ignore_index=True); metrics = pd.DataFrame(run_rows)
print(metrics.groupby(['probability_type', 'scope'])[['balanced_accuracy', 'roc_auc', 'f1', 'brier', 'ece_10bin']].mean().round(3).to_string())

seed 42 fold 0 threshold 0.339


seed 42 fold 1 threshold 0.486


seed 42 fold 2 threshold 0.48


seed 42 fold 3 threshold 0.543


seed 42 fold 4 threshold 0.698


...

seed 62 fold 0 threshold 0.859


seed 62 fold 1 threshold 0.626


seed 62 fold 2 threshold 0.412


seed 62 fold 3 threshold 0.528


seed 62 fold 4 threshold 0.65
                               balanced_accuracy  roc_auc     f1  brier  ece_10bin
probability_type scope                                                            
calibrated       felius_2024               0.813    0.920  0.868  0.091      0.144
                 pooled_all                0.837    0.944  0.865  0.091      0.112
                 voisard_2025              0.882    0.973  0.850  0.094      0.141
raw              felius_2024               0.826    0.920  0.889  0.114      0.181
                 pooled_all                0.857    0.944  0.883  0.102      0.141
                 voisard_2025              0.890    0.973  0.860  0.087      0.173


In [6]:
summary = metrics.groupby(['probability_type', 'scope']).agg({'balanced_accuracy': ['mean', 'std'], 'roc_auc': ['mean', 'std'], 'f1': ['mean', 'std'], 'brier': 'mean', 'ece_10bin': 'mean'}).reset_index()
summary.columns = ['_'.join(c).strip('_') for c in summary.columns.to_flat_index()]
print(summary.round(3).to_string(index=False))
predictions.to_csv(PROCESSED / 'repeated_pooled_outer_predictions.csv', index=False)
metrics.to_csv(PROCESSED / 'repeated_pooled_outer_metrics.csv', index=False)
summary.to_csv(PROCESSED / 'repeated_pooled_outer_summary.csv', index=False)
print('Saved repeated pooled validation outputs.')

probability_type        scope  balanced_accuracy_mean  balanced_accuracy_std  roc_auc_mean  roc_auc_std  f1_mean  f1_std  brier_mean  ece_10bin_mean
      calibrated  felius_2024                   0.813                  0.082         0.920        0.067    0.868   0.083       0.091           0.144
      calibrated   pooled_all                   0.837                  0.063         0.944        0.041    0.865   0.063       0.091           0.112
      calibrated voisard_2025                   0.882                  0.069         0.973        0.027    0.850   0.091       0.094           0.141
             raw  felius_2024                   0.826                  0.076         0.920        0.067    0.889   0.071       0.114           0.181
             raw   pooled_all                   0.857                  0.059         0.944        0.041    0.883   0.056       0.102           0.141
             raw voisard_2025                   0.890                  0.064         0.973        0.027   

## Decision gate

Use the calibrated metrics as the primary internal estimate only when the calibration split is fully inside the outer training fold. Preserve raw AUROC as a discrimination check. The model remains unsuitable for a clinical claim until it is evaluated on an untouched external cohort.